In [24]:
import pandas as pd
import math

## Cleanup

One of the largest aspects of the project (besides the algorithm itself) was the preprocessing of the data, which 
1. Standardizes (or normalizes, optionally) all numeric variables for consistent weights within the kNN algorithm, and
2. Splits the data into three parts: the training set, the evaluation set (for choosing k), and the test set.

The data points to be picked for each dataset is randomly chosen, with a seed used for reproducability. This code also allows for adjustment to the proportion of data points to be added to each split.

Reading in data + setting constants:

In [34]:
# raw data as a .csv
IMPORT_DATASET_PATH = "auto-mpg.csv"

# what are the column names you'd like to standardize?
COLUMNS_TO_STANDARDIZE = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year']

# reproducable results for a specific seed
SEED = 910202623

# how do you want to split the raw dataset up, by percentage? 
TRAINING_PCT = 0.5 # ex. 0.5 = train with 50% of the data
EVALUATING_PCT = 0.25 # ex. 0.25 = evaluate the best model (or k) with 25% of the data
TESTING_PCT = 0.25 # ex. 0.5 = test our model with the last 25% of the data
if TRAINING_PCT + EVALUATING_PCT + TESTING_PCT != 1:
    raise Exception("percentages must add to 100!")

# read raw data into pd dataframe

# column names weren't explicitly defined in the .csv downloaded (helper file only), so we do so here for easier use.
col_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year', 'origin', 'car name']
import_data = pd.read_csv(IMPORT_DATASET_PATH, sep=r'\s+', names = col_names)
IMPORT_DATA_LENGTH = len(import_data)

Standardizing + splitting data:

In [39]:
# standardize parameters to z-score (essentially)
standardized_data = import_data[[*COLUMNS_TO_STANDARDIZE, "mpg"]]

# formula
def standardize(parameter, normal = False):
    return (parameter - parameter.mean()) / (parameter.std()) if normal else (parameter - parameter.min()) / (parameter.max() - parameter.min())

# standardize each listed parameter
standardized_data[COLUMNS_TO_STANDARDIZE] = standardized_data[COLUMNS_TO_STANDARDIZE].apply(standardize, axis=0)

# split finalized dataset into three parts for training, evaluating, and testing

### translate percentages into # of rows to randomly sample
TRAINING_NUM = math.floor(TRAINING_PCT * IMPORT_DATA_LENGTH)
EVALUATING_NUM = math.floor(EVALUATING_PCT * IMPORT_DATA_LENGTH)
TESTING_NUM = IMPORT_DATA_LENGTH - TRAINING_NUM - EVALUATING_NUM

### sample # rows from finalized data and remove those rows from original df 
training_data = standardized_data.sample(n=TRAINING_NUM, random_state=SEED, replace=False)
standardized_data = standardized_data.drop(training_data.index)

evaluating_data = standardized_data.sample(n=EVALUATING_NUM, random_state=SEED, replace=False)
standardized_data = standardized_data.drop(evaluating_data.index)

### use whatever is left as the test set
testing_data = standardized_data


## DEBUGGING ##
# print(len(training_data))
# print(len(evaluating_data))
# print(len(testing_data))
# print("should equal...")
# print(TRAINING_NUM)
# print(EVALUATING_NUM)
# print(TESTING_NUM)
###############


Exporting data:

In [41]:
# export final sets to .csv
training_data.to_csv(f"{IMPORT_DATASET_PATH.replace(".csv", "_")}training_data.csv", index=False)
evaluating_data.to_csv(f"{IMPORT_DATASET_PATH.replace(".csv", "_")}evaluating_data.csv", index=False)
testing_data.to_csv(f"{IMPORT_DATASET_PATH.replace(".csv", "_")}testing_data.csv", index=False)